# Two-video room mesh: VGGT → COLMAP BA → CUDA MVS → mesh

This notebook starts with the 48-frame cross-video pilot. Do not run a larger tier until the pilot registers frames from both videos into one coherent reconstruction.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/room_reconstruction')
INPUT_ZIP = DRIVE_ROOT / 'room_mesh_colab_input.zip'
WORK_ROOT = Path('/content/room_reconstruction')
VGGT_DIR = Path('/content/vggt')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
assert INPUT_ZIP.is_file(), f'Upload the prepared zip here first: {INPUT_ZIP}'
print('Input:', INPUT_ZIP, round(INPUT_ZIP.stat().st_size / 1e6, 1), 'MB')


In [ ]:
import subprocess, sys, torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'
gpu_name = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'GPU: {gpu_name} ({gpu_gb:.1f} GB)')
if not VGGT_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/facebookresearch/vggt.git', str(VGGT_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(VGGT_DIR / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(VGGT_DIR / 'requirements_demo.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pycolmap-cuda12', 'trimesh', 'matplotlib'], check=True)


In [ ]:
import json, shutil, zipfile
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)
with zipfile.ZipFile(INPUT_ZIP) as archive:
    archive.extractall(WORK_ROOT)
manifest = json.loads((WORK_ROOT / 'manifest.json').read_text())
print(json.dumps(manifest['scenes'], indent=2))
print('Usable filtered frames:', manifest['usable_frames'])


## Pilot: jointly solve both videos
The pilot is deliberately small enough for a T4. It is a validation gate, not the final mesh.

In [ ]:
import subprocess, sys
def run_vggt_colmap(scene_dir, query_frames=8, query_points=2048):
    sparse = scene_dir / 'sparse'
    if (sparse / 'images.bin').is_file():
        print('Reusing', sparse)
        return
    command = [
        sys.executable, 'demo_colmap.py', f'--scene_dir={scene_dir}', '--use_ba',
        f'--max_query_pts={query_points}', f'--query_frame_num={query_frames}',
    ]
    print(' '.join(map(str, command)))
    subprocess.run(command, cwd=VGGT_DIR, check=True)

PILOT_SCENE = WORK_ROOT / 'scene_pilot'
run_vggt_colmap(PILOT_SCENE)


In [ ]:
import collections, numpy as np, matplotlib.pyplot as plt, pycolmap
def validate_reconstruction(scene_dir, minimum_ratio=0.80):
    reconstruction = pycolmap.Reconstruction(str(scene_dir / 'sparse'))
    input_count = len(list((scene_dir / 'images').glob('*.jpg')))
    registered = list(reconstruction.images.values())
    ratio = len(registered) / max(1, input_count)
    by_video = collections.Counter(image.name.split('_', 1)[0] for image in registered)
    print('Registered:', len(registered), '/', input_count, f'({ratio:.1%})')
    print('By source video:', dict(by_video))
    print('Sparse points:', len(reconstruction.points3D))
    assert ratio >= minimum_ratio, 'Registration ratio is too low; do not continue.'
    assert by_video.get('01', 0) > 0 and by_video.get('02', 0) > 0, 'One video did not join the reconstruction.'
    centers = []
    for image in registered:
        pose = image.cam_from_world() if callable(image.cam_from_world) else image.cam_from_world
        centers.append(np.asarray(pose.inverse().translation))
    centers = np.asarray(centers)
    plt.figure(figsize=(7, 7))
    plt.scatter(centers[:, 0], centers[:, 2], s=12, alpha=.75)
    plt.axis('equal'); plt.grid(alpha=.2); plt.title('Pilot camera centers — inspect for one coherent room')
    plt.show()
    return reconstruction
pilot_reconstruction = validate_reconstruction(PILOT_SCENE)


## Larger solve (run only after the pilot looks correct)
T4 defaults to the pilot, L4/24 GB to the 96-frame medium set, and A100/40+ GB to the 220-frame full set. Override `SCENE_TIER` only if you understand the memory tradeoff.

In [ ]:
RUN_LARGER = False  # change to True only after the pilot passes
SCENE_TIER = 'scene_full' if gpu_gb >= 38 else ('scene_medium' if gpu_gb >= 22 else 'scene_pilot')
ACTIVE_SCENE = WORK_ROOT / SCENE_TIER
print('Selected tier:', SCENE_TIER)
if RUN_LARGER:
    run_vggt_colmap(ACTIVE_SCENE, query_frames=8, query_points=4096 if gpu_gb >= 22 else 2048)
    active_reconstruction = validate_reconstruction(ACTIVE_SCENE, minimum_ratio=0.85)
else:
    ACTIVE_SCENE = PILOT_SCENE
    active_reconstruction = pilot_reconstruction


## CUDA dense reconstruction and Poisson mesh
Run this first on the validated pilot. PatchMatch requires a CUDA-enabled PyCOLMAP build.

In [ ]:
RUN_DENSE = False  # change to True after inspecting the trajectory above
if RUN_DENSE:
    dense = ACTIVE_SCENE / 'dense'
    fused = ACTIVE_SCENE / 'fused.ply'
    raw_mesh = ACTIVE_SCENE / 'room_poisson_raw.ply'
    web_mesh = ACTIVE_SCENE / 'room_poisson_web.ply'
    pycolmap.undistort_images(
        output_path=dense, input_path=ACTIVE_SCENE / 'sparse', image_path=ACTIVE_SCENE / 'images',
        undistort_options=pycolmap.UndistortCameraOptions(max_image_size=1600),
    )
    pycolmap.patch_match_stereo(
        dense, options=pycolmap.PatchMatchOptions(max_image_size=1600, gpu_index='0', cache_size=min(16.0, gpu_gb * .55)),
    )
    pycolmap.stereo_fusion(
        fused, dense, input_type='geometric', output_type='PLY',
        options=pycolmap.StereoFusionOptions(max_image_size=1600, cache_size=min(16.0, gpu_gb * .55)),
    )
    pycolmap.poisson_meshing(fused, raw_mesh, pycolmap.PoissonMeshingOptions(depth=11, trim=9.0, color=True))
    import trimesh
    mesh = trimesh.load(raw_mesh, process=False)
    ratio = min(1.0, 200000 / max(1, len(mesh.faces)))
    pycolmap.simplify_mesh(raw_mesh, web_mesh, pycolmap.MeshSimplificationOptions(target_face_ratio=ratio, interpolate_colors=True))
    print('Dense cloud:', fused, round(fused.stat().st_size / 1e6, 1), 'MB')
    print('Web mesh:', web_mesh, round(web_mesh.stat().st_size / 1e6, 1), 'MB')


In [ ]:
# Preserve the active reconstruction and previews in Drive.
RESULT_ZIP = DRIVE_ROOT / f'{ACTIVE_SCENE.name}_vggt_colmap_mesh.zip'
if RUN_DENSE:
    shutil.make_archive(str(RESULT_ZIP.with_suffix('')), 'zip', ACTIVE_SCENE)
    print('Saved:', RESULT_ZIP)
else:
    print('Set RUN_DENSE=True after the pilot trajectory is verified.')
